# Lab 9: Histogram Equalization
Chọn **Runtime → Change runtime type → T4 GPU**, rồi **Run all**.
Có thể tải `image.jpg` vào Files trước khi chạy; nếu không có, notebook dùng ảnh mẫu có sẵn.

Kết quả: ảnh, `metrics.csv`, báo cáo LaTeX và file `outputs/Lab9.zip`.
Thời gian GPU đã warm-up, đồng bộ; không tính truyền dữ liệu. Dùng GPU thật để đo speedup.


In [ ]:
%pip -q install "numba-cuda[cu12]"


In [ ]:
import time
import shutil
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from numba import cuda, float32, float64

OUT = Path('outputs/Lab9')
OUT.mkdir(parents=True, exist_ok=True)

assert cuda.is_available(), 'Chọn Runtime > Change runtime type > T4 GPU rồi chạy lại.'
device = cuda.get_current_device()
print('GPU:', device.name)
MAX_THREADS = device.MAX_THREADS_PER_BLOCK
BLOCKS = [b for b in [(8, 8), (16, 16), (32, 16), (32, 32)]
          if b[0] * b[1] <= MAX_THREADS]
REPEATS = 3

def load_image(path='image.jpg'):
    if Path(path).exists():
        a = plt.imread(path)
        if np.issubdtype(a.dtype, np.floating):
            a = np.clip(a * 255, 0, 255).astype(np.uint8)
        return np.asarray(Image.fromarray(a).convert('RGB')).copy()
    # Ảnh mẫu có màu, cạnh và nhiễu; không cần tải file để Run all.
    y, x = np.indices((256, 256))
    a = np.stack((x, y, (x // 32 % 2) * 160 + 40), axis=-1)
    noise = np.random.default_rng(0).integers(-15, 16, a.shape)
    return np.clip(a + noise, 0, 255).astype(np.uint8)

image = load_image()
H, W = image.shape[:2]
gray = (0.299 * image[:, :, 0] + 0.587 * image[:, :, 1]
        + 0.114 * image[:, :, 2]).astype(np.uint8)
print('Ảnh:', image.shape)

def grid(block):
    return ((W + block[0] - 1) // block[0],
            (H + block[1] - 1) // block[1])

def gpu_time(run):
    run()  # Warm-up: không tính biên dịch JIT vào thời gian.
    cuda.synchronize()
    start = time.time()
    for _ in range(REPEATS):
        run()
    cuda.synchronize()
    return (time.time() - start) / REPEATS

def cpu_time(run):
    start = time.time()
    result = run()
    return result, time.time() - start

def show_images(items):
    fig, axes = plt.subplots(1, len(items), figsize=(4 * len(items), 4), squeeze=False)
    for ax, (title, a) in zip(axes[0], items):
        ax.imshow(a, cmap='gray', vmin=0, vmax=255)
        ax.set_title(title)
        ax.axis('off')
    fig.tight_layout()
    fig.savefig(OUT / 'images.png', dpi=120)
    plt.show()

def save_image(name, a):
    Image.fromarray(a).save(OUT / name)

def save_results(rows, report_name, report_text):
    df = pd.DataFrame(rows)
    df.to_csv(OUT / 'metrics.csv', index=False)
    print(df.to_string(index=False))
    # Bảng LaTeX nhỏ, không cần thêm thư viện dựng báo cáo.
    def tex(v):
        return str(v).replace('_', r'\_')
    lines = [r'\begin{center}', r'\resizebox{\linewidth}{!}{',
             r'\begin{tabular}{' + 'l' * len(df.columns) + '}', r'\hline',
             ' & '.join(tex(c) for c in df.columns) + r' \\', r'\hline']
    for row in df.itertuples(index=False, name=None):
        lines.append(' & '.join(f'{v:.6g}' if isinstance(v, float) else tex(v)
                                for v in row) + r' \\')
    lines += [r'\hline', r'\end{tabular}', '}', r'\end{center}']
    (OUT / 'results.tex').write_text('\n'.join(lines), encoding='utf-8')
    (OUT / report_name).write_text(report_text, encoding='utf-8')
    shutil.make_archive(str(OUT), 'zip', OUT)
    print('Kết quả và báo cáo:', str(OUT) + '.zip (tải trong Files bên trái)')


In [ ]:
@cuda.jit
def histogram(src, hist):
    x, y = cuda.grid(2)
    if x < src.shape[1] and y < src.shape[0]:
        cuda.atomic.add(hist, int(src[y, x]), 1)

@cuda.jit
def equalize(src, dst, lut):
    x, y = cuda.grid(2)
    if x < src.shape[1] and y < src.shape[0]:
        dst[y, x] = lut[src[y, x]]


In [ ]:
d_src = cuda.to_device(gray)
zeros = np.zeros(256, dtype=np.int32)
d_hist = cuda.to_device(zeros)
block = (16, 16)
# Histogram được reset trước mỗi lần đo, không cộng dồn qua warm-up/repeats.
histogram[grid(block), block](d_src, d_hist); cuda.synchronize()
hist_s = 0.0
for _ in range(REPEATS):
    d_hist.copy_to_device(zeros); cuda.synchronize()
    start = time.time()
    histogram[grid(block), block](d_src, d_hist); cuda.synchronize()
    hist_s += (time.time() - start) / REPEATS
hist = d_hist.copy_to_host()
assert np.array_equal(hist, np.bincount(gray.ravel(), minlength=256))
assert hist.sum() == H * W
cdf = hist.cumsum()
cdf_min = cdf[hist > 0][0]
lut = np.arange(256, dtype=np.uint8)  # Ảnh hằng: giữ nguyên, tránh chia 0.
if H * W > cdf_min:
    lut = np.clip(np.rint((cdf - cdf_min) * 255.0 / (H * W - cdf_min)), 0, 255).astype(np.uint8)
d_lut = cuda.to_device(lut)
d_dst = cuda.device_array_like(d_src)
map_s = gpu_time(lambda: equalize[grid(block), block](d_src, d_dst, d_lut))
result = d_dst.copy_to_host()
assert np.array_equal(result, lut[gray])
after = np.bincount(result.ravel(), minlength=256)
pd.DataFrame(dict(intensity=np.arange(256), before=hist, after=after)).to_csv(OUT / 'histogram.csv', index=False)
rows = [dict(histogram_ms=hist_s * 1000, equalization_ms=map_s * 1000,
             pixels=int(hist.sum()), max_error=0)]
save_image('image_equalized.png', result)
show_images([('Input', gray), ('Equalized', result)])
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
for ax, title, counts in zip(axes, ['Before', 'After'], [hist, after]):
    ax.bar(np.arange(256), counts, width=1); ax.set_title(title)
    ax.set_xlabel('Intensity'); ax.set_ylabel('Pixels')
fig.tight_layout(); fig.savefig(OUT / 'histograms.png', dpi=120); plt.show()


In [ ]:
report_text = r"""
\documentclass{article}
\usepackage[utf8]{inputenc}
\usepackage[T5]{fontenc}
\usepackage{graphicx}
\usepackage[margin=1.5cm]{geometry}
\title{Lab 9: Histogram Equalization}
\author{Nguyễn Thanh Hùng -- 2540056}
\date{}
\begin{document}
\maketitle
A 2D kernel counts 256 grayscale bins using atomic additions. The histogram is reset before each timed run.
The CPU builds the CDF lookup table using the first occupied CDF value; a CUDA map applies it. Constant images are preserved.
The notebook validates counts and mapping, saves histogram CSV data, and plots before/after histograms. Histogram and mapping times are separate; CDF construction is on the CPU.

Kernel timing uses \texttt{time.time()} after JIT warm-up and CUDA synchronization,
averaged over three GPU runs. Device allocation and host/device copies are excluded.
CPU measurements are single runs. Speedup is relative to the stated CPU implementation.
Actual measurements appear after running the notebook on Colab; no timings are assumed here.
\IfFileExists{results.tex}{\input{results.tex}}{}
\IfFileExists{performance.png}{\begin{center}\includegraphics[width=.75\linewidth]{performance.png}\end{center}}{}
\IfFileExists{images.png}{\begin{center}\includegraphics[width=\linewidth]{images.png}\end{center}}{}
\IfFileExists{histograms.png}{\begin{center}\includegraphics[width=.8\linewidth]{histograms.png}\end{center}}{}
\end{document}
"""
save_results(rows, 'Report.9.gather.tex', report_text)
